<a href="https://colab.research.google.com/github/chawlaniharika23/Fraud_Detection_Research/blob/main/notebooks/02_data_preprocessing_and_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2: Data Preprocessing and Feature Engineering

## Objective
This notebook prepares the IEEE-CIS and PaySim financial fraud datasets for machine-learning experiments.

The preprocessing steps include:
- Loading the datasets
- Removing duplicate records
- Examining missing values
- Identifying and handling outliers
- Encoding categorical variables
- Scaling numerical features where required
- Creating temporal and behavioural features
- Preventing data leakage

The IEEE-CIS and PaySim datasets will be processed separately because they have different feature structures.

## Expected outcome
Cleaned and prepared datasets that can be used for feature selection and model development in the next notebooks.

In [ ]:
# Overview: Import the libraries needed for data loading, preprocessing, splitting, and memory management.
import pandas as pd
import numpy as np
import os
import gc

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Overview: Load the IEEE-CIS transaction and identity files, then merge them using TransactionID.
import pandas as pd
import numpy as np
import os
import gc

DATA_PATH = "/content/drive/MyDrive/fraud-detection-research/data"

IEEE_TRANSACTION_FILE = os.path.join(DATA_PATH, "train_transaction.csv")
IEEE_IDENTITY_FILE = os.path.join(DATA_PATH, "train_identity.csv")
PAYSIM_FILE = os.path.join(
    DATA_PATH, "PS_20174392719_1491204439457_log.csv"
)

# Load IEEE-CIS
ieee_transaction = pd.read_csv(IEEE_TRANSACTION_FILE)
ieee_identity = pd.read_csv(IEEE_IDENTITY_FILE)

ieee_data = ieee_transaction.merge(
    ieee_identity,
    on="TransactionID",
    how="left",
    validate="one_to_one"
)

print("IEEE-CIS shape:", ieee_data.shape)

del ieee_transaction, ieee_identity
gc.collect()

IEEE-CIS shape: (590540, 434)


0

In [ ]:
#Overview: Save the merged IEEE-CIS dataset in Parquet format and release the large in-memory dataframe.
PROCESSED_PATH = os.path.join(DATA_PATH, "processed")
os.makedirs(PROCESSED_PATH, exist_ok=True)

ieee_data.to_parquet(
    os.path.join(PROCESSED_PATH, "ieee_merged.parquet"),
    index=False
)

print("IEEE-CIS saved successfully.")

del ieee_data
gc.collect()

IEEE-CIS saved successfully.


30

In [ ]:
# Overview: Load the PaySim transaction dataset and display its size.paysim_data = pd.read_csv(
    PAYSIM_FILE,
    low_memory=False
)

print("PaySim shape:", paysim_data.shape)
print("PaySim loaded successfully.")

PaySim shape: (6362620, 11)
PaySim loaded successfully.


In [ ]:
# Overview: Save the PaySim dataset in Parquet format for faster future loading.
paysim_data.to_parquet(
    os.path.join(PROCESSED_PATH, "paysim.parquet"),
    index=False
)

print("PaySim saved successfully.")

PaySim saved successfully.


In [ ]:
# Overview: Delete the raw PaySim dataframe and free memory.
del paysim_data
gc.collect()

print("PaySim memory released.")

PaySim memory released.


In [ ]:
# Overview: Check that the processed IEEE-CIS and PaySim files were saved successfully.
print("IEEE-CIS saved:",
      os.path.exists(os.path.join(PROCESSED_PATH, "ieee_merged.parquet")))

print("PaySim saved:",
      os.path.exists(os.path.join(PROCESSED_PATH, "paysim.parquet")))

IEEE-CIS saved: True
PaySim saved: True


In [ ]:
# Overview: Load the merged IEEE-CIS dataset for data-quality inspection.
import pandas as pd
import numpy as np
import os
import gc

PROCESSED_PATH = "/content/drive/MyDrive/fraud-detection-research/data/processed"

ieee_clean = pd.read_parquet(
    os.path.join(PROCESSED_PATH, "ieee_merged.parquet")
)

print("IEEE-CIS loaded:", ieee_clean.shape)

IEEE-CIS loaded: (590540, 434)


In [ ]:
# Overview: Calculate missing-value counts and percentages for each IEEE-CIS column.

ieee_missing_summary = pd.DataFrame({
    "Missing_Count": ieee_clean.isnull().sum(),
    "Missing_Percentage": ieee_clean.isnull().mean() * 100
})

ieee_missing_summary = ieee_missing_summary.sort_values(
    by="Missing_Percentage",
    ascending=False
)

display(ieee_missing_summary.head(25))

print("Columns with >95% missing:",
      (ieee_missing_summary["Missing_Percentage"] > 95).sum())

print("Columns with >50% missing:",
      (ieee_missing_summary["Missing_Percentage"] > 50).sum())

print("Columns with no missing values:",
      (ieee_missing_summary["Missing_Count"] == 0).sum())

,Missing_Count,Missing_Percentage
id_24,585793,99.196159
id_25,585408,99.130965
id_07,585385,99.127070
id_08,585385,99.127070
id_21,585381,99.126393
id_26,585377,99.125715
id_27,585371,99.124699
id_23,585371,99.124699
id_22,585371,99.124699
dist2,552913,93.628374


Columns with >95% missing: 9
Columns with >50% missing: 214
Columns with no missing values: 20


In [ ]:
# Overview: Separate IEEE-CIS columns into numeric and categorical feature groups.
ieee_numeric = ieee_clean.select_dtypes(
    include=["number"]
).columns.tolist()

ieee_categorical = ieee_clean.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numeric columns:", len(ieee_numeric))
print("Categorical columns:", len(ieee_categorical))

print("\nCategorical columns:")
print(ieee_categorical)

Numeric columns: 403
Categorical columns: 31

Categorical columns:
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


In [ ]:
# Overview: Export the IEEE-CIS missing-value summary as a CSV report.
report_path = os.path.join(
    PROCESSED_PATH,
    "ieee_missing_value_report.csv"
)

ieee_missing_summary.to_csv(report_path)

print("Missing-value report saved:", report_path)

Missing-value report saved: /content/drive/MyDrive/fraud-detection-research/data/processed/ieee_missing_value_report.csv


In [ ]:
# Overview: Load the PaySim dataset and preview its records.
paysim_clean = pd.read_parquet(
    os.path.join(PROCESSED_PATH, "paysim.parquet")
)

print("PaySim shape:", paysim_clean.shape)
display(paysim_clean.head())

PaySim shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [ ]:
# Overview: List the numeric and categorical columns available in PaySim.
print("Numeric columns:")
print(paysim_clean.select_dtypes(include=["number"]).columns.tolist())

print("\nCategorical columns:")
print(paysim_clean.select_dtypes(include=["object", "category"]).columns.tolist())

Numeric columns:
['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

Categorical columns:
['type', 'nameOrig', 'nameDest']


In [ ]:
# Overview: Explore transaction-type frequencies and fraud counts by transaction type.
print("Transaction type distribution:")
display(paysim_clean["type"].value_counts())

print("\nFraud cases by transaction type:")
display(
    paysim_clean.groupby("type")["isFraud"]
    .agg(["count", "sum", "mean"])
    .sort_values("sum", ascending=False)
)

Transaction type distribution:


,count
type,
CASH_OUT,2237500
PAYMENT,2151495
CASH_IN,1399284
TRANSFER,532909
DEBIT,41432



Fraud cases by transaction type:


,count,sum,mean
type,,,
CASH_OUT,2237500,4116,0.001840
TRANSFER,532909,4097,0.007688
CASH_IN,1399284,0,0.000000
DEBIT,41432,0,0.000000
PAYMENT,2151495,0,0.000000


In [ ]:
# Overview: Inspect fraud labels, flagged-fraud labels, and the number of unique accounts.
print("Target distribution:")
print(paysim_clean["isFraud"].value_counts())

print("\nFlagged fraud distribution:")
print(paysim_clean["isFlaggedFraud"].value_counts())

print("\nUnique origin accounts:",
      paysim_clean["nameOrig"].nunique())

print("Unique destination accounts:",
      paysim_clean["nameDest"].nunique())

Target distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Flagged fraud distribution:
isFlaggedFraud
0    6362604
1         16
Name: count, dtype: int64

Unique origin accounts: 6353307
Unique destination accounts: 2722362


In [ ]:
# Create a separate feature-engineering copy
paysim_features = paysim_clean.copy()

# Transaction time features
paysim_features["hour"] = (
    paysim_features["step"] % 24
)

paysim_features["day"] = (
    paysim_features["step"] // 24
)

# Transaction type encoding
paysim_features = pd.get_dummies(
    paysim_features,
    columns=["type"],
    prefix="type",
    dtype=int
)

# Amount transformation
paysim_features["log_amount"] = np.log1p(
    paysim_features["amount"]
)

# Balance difference features
paysim_features["origin_balance_change"] = (
    paysim_features["oldbalanceOrg"]
    - paysim_features["newbalanceOrig"]
)

paysim_features["destination_balance_change"] = (
    paysim_features["newbalanceDest"]
    - paysim_features["oldbalanceDest"]
)

print("PaySim feature shape:", paysim_features.shape)
display(paysim_features.head())

PaySim feature shape: (6362620, 20)


,step,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,hour,day,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,log_amount,origin_balance_change,destination_balance_change
0,1,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,1,0,0,0,0,1,0,9.194276,9839.64,0.0
1,1,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,1,0,0,0,0,1,0,7.531166,1864.28,0.0
2,1,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,1,0,0,0,0,0,1,5.204007,181.00,0.0
3,1,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,1,0,0,1,0,0,0,5.204007,181.00,-21182.0
4,1,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,1,0,0,0,0,1,0,9.364703,11668.14,0.0


In [ ]:
# Overview: Separate the PaySim fraud target from the input features and remove identifiers or target-related columns.
paysim_target = paysim_features["isFraud"].copy()

paysim_X = paysim_features.drop(
    columns=[
        "isFraud",
        "isFlaggedFraud",
        "nameOrig",
        "nameDest",
        "step"
    ],
    errors="ignore"
)

print("PaySim feature matrix:", paysim_X.shape)
print("PaySim target:", paysim_target.shape)
print("\nRemaining feature columns:")
print(paysim_X.columns.tolist())

PaySim feature matrix: (6362620, 15)
PaySim target: (6362620,)

Remaining feature columns:
['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'hour', 'day', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER', 'log_amount', 'origin_balance_change', 'destination_balance_change']


In [ ]:
# Overview: Create real-time and extended PaySim feature sets for separate experiments.
# Feature set A: stricter real-time prediction features
paysim_X_realtime = paysim_X.drop(
    columns=[
        "newbalanceOrig",
        "oldbalanceDest",
        "newbalanceDest",
        "origin_balance_change",
        "destination_balance_change"
    ]
)

# Feature set B: expanded experimental feature set
paysim_X_extended = paysim_X.copy()

print("Real-time feature set:", paysim_X_realtime.shape)
print("Extended feature set:", paysim_X_extended.shape)

Real-time feature set: (6362620, 10)
Extended feature set: (6362620, 15)


In [ ]:
# Overview: Save both PaySim feature sets and the fraud target as Parquet files.
paysim_X_realtime.to_parquet(
    os.path.join(PROCESSED_PATH, "paysim_realtime_features.parquet"),
    index=False
)

paysim_X_extended.to_parquet(
    os.path.join(PROCESSED_PATH, "paysim_extended_features.parquet"),
    index=False
)

paysim_target.to_frame("isFraud").to_parquet(
    os.path.join(PROCESSED_PATH, "paysim_target.parquet"),
    index=False
)

print("PaySim feature sets and target saved successfully.")

PaySim feature sets and target saved successfully.


In [ ]:
## Overview: Remove temporary PaySim dataframes and release memory.
del paysim_clean
del paysim_features
del paysim_X
del paysim_X_realtime
del paysim_X_extended
del paysim_target

gc.collect()

print("PaySim working variables released.")

PaySim working variables released.


In [ ]:
# Overview: Reload the merged IEEE-CIS dataset for feature preparation.

ieee_clean = pd.read_parquet(
    os.path.join(PROCESSED_PATH, "ieee_merged.parquet")
)

print("IEEE-CIS loaded:", ieee_clean.shape)

IEEE-CIS loaded: (590540, 434)


In [ ]:
# Overview: Separate the IEEE-CIS fraud label from the model input features.
ieee_target = ieee_clean["isFraud"].copy()

# Exclude target and transaction identifier
ieee_X = ieee_clean.drop(
    columns=["isFraud", "TransactionID"]
)

print("IEEE-CIS feature matrix:", ieee_X.shape)
print("IEEE-CIS target:", ieee_target.shape)

IEEE-CIS feature matrix: (590540, 432)
IEEE-CIS target: (590540,)


In [ ]:
# Overview: Identify the categorical features in the IEEE-CIS input data.

ieee_categorical = ieee_X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Number of categorical features:", len(ieee_categorical))
print(ieee_categorical)

Number of categorical features: 31
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


In [ ]:
# Overview: Measure the number of unique and missing values in each categorical feature.
ieee_cardinality = pd.DataFrame({
    "Unique_Values": ieee_X[ieee_categorical].nunique(),
    "Missing_Values": ieee_X[ieee_categorical].isnull().sum()
}).sort_values(
    by="Unique_Values",
    ascending=False
)

display(ieee_cardinality)

,Unique_Values,Missing_Values
DeviceInfo,1786,471874
id_33,260,517251
id_31,130,450258
id_30,75,512975
R_emaildomain,60,453249
P_emaildomain,59,94456
ProductCD,5,0
card6,4,1571
card4,4,1577
id_34,4,512735


In [ ]:
# Overview: Group categorical features into low- and high-cardinality columns.
# Select categorical fields with at most 50 unique values
low_cardinality_cols = [
    col for col in ieee_categorical
    if ieee_X[col].nunique(dropna=True) <= 50
]

high_cardinality_cols = [
    col for col in ieee_categorical
    if col not in low_cardinality_cols
]

print("Low-cardinality columns:", len(low_cardinality_cols))
print(low_cardinality_cols)

print("\nHigh-cardinality columns:", len(high_cardinality_cols))
print(high_cardinality_cols)

Low-cardinality columns: 25
['ProductCD', 'card4', 'card6', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType']

High-cardinality columns: 6
['P_emaildomain', 'R_emaildomain', 'id_30', 'id_31', 'id_33', 'DeviceInfo']


In [ ]:
# Overview: Save categorical feature-cardinality details and feature-group reports.
ieee_cardinality.to_csv(
    os.path.join(PROCESSED_PATH, "ieee_categorical_cardinality.csv")
)

pd.DataFrame({
    "Low_Cardinality_Features": pd.Series(low_cardinality_cols),
    "High_Cardinality_Features": pd.Series(high_cardinality_cols)
}).to_csv(
    os.path.join(PROCESSED_PATH, "ieee_categorical_feature_groups.csv"),
    index=False
)

print("Categorical analysis reports saved.")

Categorical analysis reports saved.


In [ ]:
# Overview: Split IEEE-CIS into stratified training, validation, and test sets using a 70/15/15 ratio.

from sklearn.model_selection import train_test_split

# First split: 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    ieee_X,
    ieee_target,
    test_size=0.30,
    random_state=42,
    stratify=ieee_target
)

# Second split: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Testing set:", X_test.shape, y_test.shape)

print("\nFraud percentages:")
print("Train:", round(y_train.mean() * 100, 3))
print("Validation:", round(y_val.mean() * 100, 3))
print("Test:", round(y_test.mean() * 100, 3))

Training set: (413378, 432) (413378,)
Validation set: (88581, 432) (88581,)
Testing set: (88581, 432) (88581,)

Fraud percentages:
Train: 3.499
Validation: 3.5
Test: 3.498


In [ ]:
# Overview: Save each IEEE-CIS feature split and its corresponding fraud labels separately.
# Save features and labels separately
for split_name, X_split, y_split in [
    ("train", X_train, y_train),
    ("validation", X_val, y_val),
    ("test", X_test, y_test)
]:
    X_split.to_parquet(
        os.path.join(PROCESSED_PATH, f"ieee_X_{split_name}.parquet"),
        index=False
    )

    y_split.to_frame("isFraud").to_parquet(
        os.path.join(PROCESSED_PATH, f"ieee_y_{split_name}.parquet"),
        index=False
    )

print("All IEEE-CIS splits saved successfully.")

All IEEE-CIS splits saved successfully.


In [ ]:
# Overview: Verify that all six saved IEEE-CIS split files exist.
expected_files = [
    "ieee_X_train.parquet",
    "ieee_y_train.parquet",
    "ieee_X_validation.parquet",
    "ieee_y_validation.parquet",
    "ieee_X_test.parquet",
    "ieee_y_test.parquet"
]

for filename in expected_files:
    path = os.path.join(PROCESSED_PATH, filename)
    print(filename, ":", os.path.exists(path))

ieee_X_train.parquet : True
ieee_y_train.parquet : True
ieee_X_validation.parquet : True
ieee_y_validation.parquet : True
ieee_X_test.parquet : True
ieee_y_test.parquet : True


In [ ]:
# Overview: Delete large temporary IEEE-CIS dataframes and release memory.
del ieee_clean, ieee_X, ieee_target
del X_temp, y_temp

gc.collect()

print("Unnecessary IEEE-CIS copies released.")

Unnecessary IEEE-CIS copies released.


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
# Overview: Import the preprocessing tools used for imputation, encoding, and scaling.
numeric_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

low_cardinality_cols = [
    col for col in categorical_cols
    if X_train[col].nunique(dropna=True) <= 50
]

high_cardinality_cols = [
    col for col in categorical_cols
    if col not in low_cardinality_cols
]

print("Numeric features:", len(numeric_cols))
print("Low-cardinality categorical:", len(low_cardinality_cols))
print("High-cardinality categorical:", len(high_cardinality_cols))

Numeric features: 401
Low-cardinality categorical: 25
High-cardinality categorical: 6


In [ ]:
# Overview: Identify numeric, low-cardinality categorical, and high-cardinality features from the training data.

def frequency_encode(train_series, other_series_list):
    frequencies = train_series.value_counts(normalize=True, dropna=True)

    train_encoded = train_series.map(frequencies).fillna(0)

    other_encoded = [
        series.map(frequencies).fillna(0)
        for series in other_series_list
    ]

    return train_encoded, other_encoded

In [ ]:
# Overview: Define a frequency-encoding function that learns category frequencies from training data.
frequency_maps = {}

for col in high_cardinality_cols:
    train_values, other_values = frequency_encode(
        X_train[col],
        [X_val[col], X_test[col]]
    )

    X_train.loc[:, col] = train_values
    X_val.loc[:, col] = other_values[0]
    X_test.loc[:, col] = other_values[1]

    frequency_maps[col] = X_train[col].value_counts().to_dict()

print("Frequency encoding completed.")

Frequency encoding completed.


In [ ]:
# Overview: Apply training-derived frequency encoding to high-cardinality columns in all data splits.

numeric_imputer = SimpleImputer(strategy="median")

X_train_numeric = numeric_imputer.fit_transform(
    X_train[numeric_cols]
)

X_val_numeric = numeric_imputer.transform(
    X_val[numeric_cols]
)

X_test_numeric = numeric_imputer.transform(
    X_test[numeric_cols]
)

print("Numerical imputation completed.")
print("Training numerical shape:", X_train_numeric.shape)
print("Validation numerical shape:", X_val_numeric.shape)
print("Testing numerical shape:", X_test_numeric.shape)

Numerical imputation completed.
Training numerical shape: (413378, 401)
Validation numerical shape: (88581, 401)
Testing numerical shape: (88581, 401)


In [ ]:
# Overview: Fill missing numeric values using medians learned from the training set.
categorical_imputer = SimpleImputer(
    strategy="most_frequent"
)

X_train_cat = categorical_imputer.fit_transform(
    X_train[low_cardinality_cols]
)

X_val_cat = categorical_imputer.transform(
    X_val[low_cardinality_cols]
)

X_test_cat = categorical_imputer.transform(
    X_test[low_cardinality_cols]
)

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_encoded = encoder.fit_transform(X_train_cat)
X_val_encoded = encoder.transform(X_val_cat)
X_test_encoded = encoder.transform(X_test_cat)

print("Categorical imputation and encoding completed.")
print("Training encoded shape:", X_train_encoded.shape)
print("Validation encoded shape:", X_val_encoded.shape)
print("Testing encoded shape:", X_test_encoded.shape)

Categorical imputation and encoding completed.
Training encoded shape: (413378, 86)
Validation encoded shape: (88581, 86)
Testing encoded shape: (88581, 86)


In [ ]:
# Overview: Impute missing low-cardinality categories and one-hot encode them using the training set.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_numeric)
X_val_scaled = scaler.transform(X_val_numeric)
X_test_scaled = scaler.transform(X_test_numeric)

print("Numerical scaling completed.")
print("Training scaled shape:", X_train_scaled.shape)

Numerical scaling completed.
Training scaled shape: (413378, 401)


In [ ]:
# Overview: Standardize numeric features using statistics learned from the training set.
X_train_processed = np.hstack([
    X_train_scaled,
    X_train_encoded
])

X_val_processed = np.hstack([
    X_val_scaled,
    X_val_encoded
])

X_test_processed = np.hstack([
    X_test_scaled,
    X_test_encoded
])

print("Final processed shapes:")
print("Training:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Testing:", X_test_processed.shape)

print("Missing values in training:",
      np.isnan(X_train_processed).sum())

Final processed shapes:
Training: (413378, 487)
Validation: (88581, 487)
Testing: (88581, 487)
Missing values in training: 0


In [ ]:
# Overview: Combine scaled numeric and encoded categorical features into final model-ready matrices
import os

folder = "/content/drive/MyDrive/fraud-detection-research/data/processed"

for name in sorted(os.listdir(folder)):
    path = os.path.join(folder, name)
    print(name, f"({os.path.getsize(path) / (1024**2):.1f} MB)")

ieee_X_test.parquet (13.2 MB)
ieee_X_test_processed.joblib (329.1 MB)
ieee_X_train.parquet (59.3 MB)
ieee_X_train_processed.joblib (1040.0 MB)
ieee_X_validation.parquet (13.2 MB)
ieee_X_validation_processed.joblib (329.1 MB)
ieee_categorical_cardinality.csv (0.0 MB)
ieee_categorical_feature_groups.csv (0.0 MB)
ieee_merged.parquet (80.8 MB)
ieee_missing_value_report.csv (0.0 MB)
ieee_y_test.joblib (2.7 MB)
ieee_y_test.parquet (0.0 MB)
ieee_y_train.joblib (12.6 MB)
ieee_y_train.parquet (0.0 MB)
ieee_y_validation.joblib (2.7 MB)
ieee_y_validation.parquet (0.0 MB)
paysim.parquet (257.1 MB)
paysim_extended_features.parquet (252.7 MB)
paysim_realtime_features.parquet (116.7 MB)
paysim_target.parquet (0.0 MB)


In [ ]:
# Overview: List processed-data files and display their sizes to inspect saved outputs.

import os

path = "/content/drive/MyDrive/fraud-detection-research/data/processed/ieee_X_train_processed.joblib"

print("Training file exists:", os.path.exists(path))
print("File size (MB):", round(os.path.getsize(path) / (1024**2), 2))

Training file exists: True
File size (MB): 1040.0
